# Lab 6.1 &mdash; A Retriever You Can Inspect

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 2 &middot; Module 6 &mdash; Agentic RAG**

### What you'll do
- Split a document two ways with real splitters, and watch one of them make an answer unreachable
- Write an <code>Embeddings</code> class and index the corpus in <strong>Chroma</strong>
- See top-k return k whatever is in the corpus &mdash; then choose the floor that makes an empty result possible
- Scope with a metadata filter, which is what production retrieval actually looks like

> **How this lab works.** You write real LangChain code. Fill every `BLANK`, then run the
> **Self-check** cell under each section &mdash; those check the *objects you built* (a chunked
> `Document`, a Chroma collection, a bound tool, a compiled graph, a parser), so they are
> deterministic and do not depend on the model. Cells marked **Run it for real** put your code in
> front of the sandbox model; that is the part worth watching. The score line is feedback, not a
> grade.

> **Everything here runs offline.** The embedding model is thirty lines you can read,
> not an 80&nbsp;MB download &mdash; the sandbox has no egress. Chunking, ranking, floors
> and filters are the same whatever computes the similarity, and they decide more than
> the model does.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-6-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Thinking is off by default here because you will make a lot of calls today;
# pass think=True to any call below to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the corpus (synthetic, self-contained)
# Two short operating documents about the same payments. Read 3.2: the rule and the exception
# that qualifies it are adjacent sentences, which is the whole of Lab 6.1's first lesson. Note
# also what is NOT here -- nothing mentions FX or hedging anywhere, and Lab 6.4 needs that gap.

DOCS = {
    "ops-runbook-v4.md": """## 3.1 Insufficient funds
A payment returned INSUFFICIENT_FUNDS is retried once after 24 hours. If the retry also fails,
notify the client desk. Operations must not fund the account manually.

## 3.2 Limit breaches
Payments above USD 500,000 require Treasury approval before release. This does not apply to
intra-group transfers, which settle same-day without any approval.

## 3.3 Invalid beneficiary details
A payment returned INVALID_IBAN is returned to the originator with code R04. Beneficiary
details are never repaired in-house.

## 3.4 Sanctions review
A payment held for SANCTIONS_REVIEW is decided by Compliance. Operations must not release or
cancel it under any circumstances.
""",
    "escalation-policy-v2.md": """## 1 Approval authority
A duty manager may approve a release up to USD 250,000. Above that figure Treasury approval is
required, and must be recorded against the payment reference.

## 2 Escalation timers
If an approver has not responded within 15 minutes, escalate to the Treasury lead, and after a
further 15 minutes to the head of operations.
""",
}

print(f"{len(DOCS)} documents, {sum(len(d) for d in DOCS.values())} characters")

## Concept

A retriever is four decisions, and only one of them is the embedding model:

| Decision | The LangChain piece |
|---|---|
| how the corpus is cut up | a **text splitter** &mdash; decides what can ever be returned together |
| how a chunk is scored | an **`Embeddings`** implementation |
| how many come back, and how bad they may be | `k`, and a **floor** you impose yourself |
| what is in scope before ranking starts | a **metadata filter** |

You build all four in this lab, on a real `Chroma` collection. Three of them are yours to
decide; only the second is bought off a shelf, and it is the one people think is the whole thing.

## Section 1 &mdash; Chunking decides what can be found

Section 3.2 states a rule and then exempts intra-group transfers from it. Cut that in half and no
retriever can ever return the two together, because they are no longer one thing.

`MarkdownHeaderTextSplitter` splits on headings and writes the heading into each chunk's
metadata. `RecursiveCharacterTextSplitter` splits on size, and does not care what it cuts.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

def split_by_section(doc_name: str, text: str) -> list:
    """One Document per '##' section, so a rule and its exception stay together."""
    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[("##", "section")],
        # TODO: the heading "3.2 Limit breaches" carries words a query will use. Should it stay
        # in the text that gets embedded, or be stripped out and left only in the metadata?
        strip_headers=BLANK)
    chunks = splitter.split_text(text)
    for chunk in chunks:
        chunk.metadata["source"] = doc_name      # the splitter fills in "section"; we add the file
    return chunks


def split_by_size(doc_name: str, text: str, size: int = 120) -> list:
    """The naive alternative: cut every `size` characters, meaning be damned."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=0)
    return splitter.create_documents([text], metadatas=[{"source": doc_name}])


def build_chunks(splitter_fn) -> list:
    """Run one splitter over every document in the corpus."""
    return [c for name, text in DOCS.items() for c in splitter_fn(name, text)]

In [ ]:
# --- Self-check: Section 1   (Document objects only -- no store yet, no model)
def by_section():
    return build_chunks(split_by_section)

def by_size():
    return build_chunks(split_by_size)

def limit_chunk(chunks):
    """The chunk that states the USD 500,000 rule."""
    return next(c for c in chunks if "500,000" in c.page_content)

check("the section splitter finds all six sections across the two documents",
      lambda: len(by_section()) == 6)
check("each chunk is a Document that knows its file and its section",
      lambda: all(isinstance(c, Document) and c.metadata["source"] in DOCS
                  and c.metadata["section"] for c in by_section()))
check("the heading is part of the text that will be embedded, not just a label",
      lambda: "Limit breaches" in limit_chunk(by_section()).page_content,
      "a query says 'limit breach'; if that phrase is only in the metadata it cannot be matched")
check("the rule and the exception that qualifies it are in ONE chunk",
      lambda: "intra-group" in limit_chunk(by_section()).page_content)
check("cutting by size splits them apart",
      lambda: "intra-group" not in limit_chunk(by_size()).page_content,
      "after this cut, no retriever on earth can return them together")
check("and that is a boundary problem, not a small-chunk problem",
      lambda: any("intra-group" in c.page_content for c in by_size()),
      "the exception is still indexed; it is just no longer attached to the rule it qualifies")

def _compare():
    print("  by section:", limit_chunk(by_section()).page_content[:96].replace("\n", " "), "...")
    print("  by size   :", limit_chunk(by_size()).page_content[:96].replace("\n", " "), "...")
guard(_compare)

In [ ]:
# ------------------------------------------------- the embedding model (nothing to fill in)
# The sandbox has no egress, and chromadb's DEFAULT embedding function downloads about 80 MB
# of ONNX model the first time it is called. So this module brings its own: one hashed bucket
# per meaningful word, normalised to unit length. It is arithmetic rather than learning, which
# is the point -- it runs offline, it is deterministic, and you can read every line of it.
#
# What it CAN do: score two texts by the words they share. What it CANNOT do: match meaning
# with no words in common. Lab 6.2 is about living with exactly that.
import re, math, hashlib
from langchain_core.embeddings import Embeddings

STOP = set("""a an the of for is are was were do does did what which who this that these those it
its to in on at by with from about and or not no be been have has had can could should would will
you your we our i me my how why when where there here as if then than so such only just also very
more most some any other""".split())

def content_words(text: str) -> list:
    """The words worth indexing: lower-cased, no punctuation, no stop words."""
    return [w for w in re.findall(r"[a-z0-9_]+", (text or "").lower())
            if w not in STOP and len(w) > 1]


class LabEmbeddings(Embeddings):
    """A tiny embedding model you can read. Same interface as any other LangChain embedding."""

    dim = 1024                      # enough buckets that two different words rarely collide

    def _vector(self, text: str) -> list:
        vec = [0.0] * self.dim
        for word in content_words(text):
            bucket = int(hashlib.sha256(word.encode()).hexdigest()[:8], 16) % self.dim
            vec[bucket] += 1.0
        length = math.sqrt(sum(x * x for x in vec)) or 1.0
        return [x / length for x in vec]      # unit length, so cosine is just a dot product

    def embed_documents(self, texts: list) -> list:
        return [self._vector(t) for t in texts]

    def embed_query(self, text: str) -> list:
        return self._vector(text)


print("embeddings:", LabEmbeddings.dim, "dimensions, offline, deterministic")

## Section 2 &mdash; Index it in Chroma

`Chroma` is a vector store: you hand it `Document`s and an `Embeddings`, and it keeps the vectors
so you can search them. Two arguments below are worth understanding rather than copying:

- **`collection_configuration={"hnsw": {"space": "cosine"}}`** &mdash; the distance metric. In cosine
  space a distance of `0.0` means identical and `1.0` means nothing in common, so
  `similarity = 1 - distance` reads the way you expect.
- **`ids=`** &mdash; `add_documents` *upserts* on the id. Stable ids mean re-running this notebook
  leaves six chunks; ids that change mean six more every time, and every score after that is
  measured on a duplicated corpus.

In [ ]:
from langchain_chroma import Chroma

def chunk_id(chunk) -> str:
    """A stable id for one chunk -- the same string on every run of this notebook."""
    # TODO: build the id out of the chunk's own metadata, so re-running updates rather than
    # duplicates. A counter or a uuid would be different on the next run.
    return BLANK


def open_store(chunks):
    """A persistent Chroma collection over the corpus, embedded by LabEmbeddings."""
    store = Chroma(collection_name="module6-corpus",
                   embedding_function=LabEmbeddings(),
                   persist_directory=os.path.join(WORK, "chroma"),
                   collection_configuration={"hnsw": {"space": "cosine"}})
    store.add_documents(chunks, ids=[chunk_id(c) for c in chunks])
    return store


_store = None
def store():
    """The collection, built once, on first use."""
    global _store
    if _store is None:
        _store = open_store(build_chunks(split_by_section))
    return _store

In [ ]:
# --- Self-check: Section 2   (a real Chroma collection -- built locally, no network)
check("the collection holds one chunk per section",
      lambda: len(store().get()["ids"]) == 6)
check("indexing the same corpus again leaves it at six, not twelve",
      lambda: len(open_store(build_chunks(split_by_section)).get()["ids"]) == 6,
      "add_documents upserts on the id -- unstable ids duplicate the corpus on every re-run")
check("the ids are unique",
      lambda: len(set(store().get()["ids"])) == 6)
check("an id is derived from the chunk, so it is the same on a fresh split",
      lambda: chunk_id(build_chunks(split_by_section)[3])
              == chunk_id(build_chunks(split_by_section)[3]),
      "a uuid or a counter fails this, and that failure is what duplicates the corpus")
check("the metadata went into the store with the text",
      lambda: all(set(m) == {"source", "section"} for m in store().get()["metadatas"]))
check("the embedding is the one you can read, not a downloaded one",
      lambda: store().embeddings.__class__ is LabEmbeddings,
      "chromadb's default embedding function needs an 80 MB download and this sandbox has no egress")

## Section 3 &mdash; Rank, and then refuse to

`similarity_search_with_score` returns `(Document, distance)` pairs, best first. `search` below
turns those into plain dicts and drops anything under a floor.

The floor is the interesting part. **Top-k always returns k** &mdash; ask a corpus about something
it has never heard of and you still get four confident rows back. The floor is the only thing
standing between you and answering from them.

In [ ]:
def search(query: str, k: int = 4, floor: float = 0.0, where: dict | None = None) -> list:
    """Top-k from the store as plain dicts, with anything below `floor` dropped."""
    hits = store().similarity_search_with_score(query, k=k, filter=where)
    out = []
    for doc, distance in hits:
        similarity = 1.0 - distance         # cosine space: 1.0 identical, 0.0 nothing in common
        if similarity >= floor:
            out.append({"score": round(similarity, 3), "text": doc.page_content,
                        "source": doc.metadata["source"], "section": doc.metadata["section"]})
    return out


REAL_QUESTIONS = [
    "what approval does a limit breach above USD 500,000 need",
    "what happens to a payment returned INVALID_IBAN",
    "who decides on a payment held for sanctions review",
    "how much may a duty manager approve",
]
FX_Q = "what is the FX hedging policy for JPY exposure"

In [ ]:
# Look at the numbers before you pick anything. Every score, both kinds of question.
def _scores():
    for label, question in [("answerable", REAL_QUESTIONS[0]), ("not in the corpus", FX_Q)]:
        print(f"  [{label}] {question}")
        for r in search(question, k=4):
            print(f"      {r['score']:.3f}  {r['source']:26} {r['section']}")
        print()
guard(_scores)

In [ ]:
def chosen_floor() -> float:
    """The similarity a chunk must clear before it is used at all.

    The cell above printed every score. The answerable question's best chunk and the
    unanswerable question's best chunk are the two numbers your floor has to separate.
    """
    return BLANK      # TODO: pick it from those printed scores, not from a round number you like

In [ ]:
# --- Self-check: Section 3   (ranking and the floor -- still no model)
check("results come back ranked, best first",
      lambda: [r["score"] for r in search(REAL_QUESTIONS[0])]
              == sorted((r["score"] for r in search(REAL_QUESTIONS[0])), reverse=True))
check("k is respected",
      lambda: len(search(REAL_QUESTIONS[0], k=2)) == 2)
check("A QUESTION THE CORPUS CANNOT ANSWER STILL RETURNS FOUR ROWS",
      lambda: len(search(FX_Q, k=4)) == 4,
      "nothing in either document mentions FX or JPY, and four chunks come back anyway")
check("and none of them is about FX",
      lambda: not any("hedg" in r["text"].lower() for r in search(FX_Q, k=4)))
check("your floor is a similarity, somewhere between 0 and 1",
      lambda: 0.0 < chosen_floor() < 1.0)
check("your floor turns the unanswerable question into an EMPTY result",
      lambda: search(FX_Q, floor=chosen_floor()) == [],
      "this is the only thing that lets the agent say 'I could not find it'")
check("and it still answers all four real questions",
      lambda: all(search(q, floor=chosen_floor()) for q in REAL_QUESTIONS),
      "a floor that refuses everything is not a safe floor, it is a broken one")
check("a floor of 0.95 refuses even the good question -- too high is its own failure",
      lambda: search(REAL_QUESTIONS[0], floor=0.95) == [],
      "Lab 6.5 measures where the floor should sit instead of arguing about it")

## Section 4 &mdash; Scope before you rank

Most production retrieval is a metadata filter with a similarity search inside it: this version,
this jurisdiction, the documents this user is allowed to see. An unfiltered index is a disclosure
waiting to be reported.

Chroma takes the filter as `where`. One condition is a plain `{"key": value}`; two conditions
have to be spelled out with `$and` &mdash; `{"a": 1, "b": 2}` is an error, not an AND.

In [ ]:
def scope_to(doc_name: str) -> dict:
    """A filter that keeps retrieval inside ONE document."""
    # TODO: which piece of metadata did every chunk get in Section 1?
    return {BLANK: doc_name}


def scope_to_section(doc_name: str, section: str) -> dict:
    """Two conditions at once, the way Chroma wants them."""
    return {"$and": [{"source": doc_name}, {"section": section}]}

In [ ]:
# --- Self-check: Section 4   (metadata filtering -- exact, and offline)
APPROVAL_Q = "who may approve a release"

check("an unfiltered search sees both documents",
      lambda: {r["source"] for r in search(APPROVAL_Q, k=6)} == set(DOCS))
check("scoping to the escalation policy returns only its sections",
      lambda: all(r["source"] == "escalation-policy-v2.md"
                  for r in search(APPROVAL_Q, where=scope_to("escalation-policy-v2.md"))))
check("and it changes the answer, which is the whole point",
      lambda: search(APPROVAL_Q, where=scope_to("escalation-policy-v2.md"))[0]["section"]
              .startswith("1"))
check("scoping to a document that does not exist returns NOTHING, not everything",
      lambda: search(APPROVAL_Q, where=scope_to("no-such-doc.md")) == [],
      "a filter that silently falls back to the whole index is how a disclosure happens")
check("both conditions of an $and have to match",
      lambda: search(APPROVAL_Q,
                     where=scope_to_section("ops-runbook-v4.md", "no such section")) == [])
check("and a real pair matches exactly one chunk",
      lambda: len(search("limit breach", k=6,
                         where=scope_to_section("ops-runbook-v4.md", "3.2 Limit breaches"))) == 1)

## Run it for real &mdash; the chunking decides the answer

One question, one model, two chunkings. The context is the chunk that states the USD 500,000
rule &mdash; taken once from the section split and once from the size split.

In [ ]:
if llm_ready():
    def _chunking_changes_the_answer():
        question = "Can we release a large intra-group transfer without Treasury approval?"
        for label, chunks in (("cut on section headings", build_chunks(split_by_section)),
                              ("cut every 120 characters", build_chunks(split_by_size))):
            context = limit_chunk(chunks).page_content
            reply = ask(f"Context:\n{context}\n\nQuestion: {question}\n\n"
                        "Answer from the context alone, in one sentence.",
                        system="Be brief. If the context does not settle it, say so.")
            print(f"  [{label}]")
            print(f"      context: {context[:88]}...".replace("\n", " "))
            print(f"      answer : {reply.strip()[:200]}")
            print()
    guard(_chunking_changes_the_answer)

### Read it

The model is the same in both halves. The retrieval floor, the metadata, the prompt and the
question are the same. The only difference is where a splitter put a boundary &mdash; and the
size-split context stops one clause short of *&ldquo;this does not apply to intra-group transfers&rdquo;*,
so a correct-sounding answer is now the wrong one. No amount of prompting fixes that: the words
are not in front of the model.

Two things to carry out of this lab.

**The floor is a policy, not a constant.** You chose a number that separates a question the corpus
answers from one it does not. The gap on this corpus is wide, because the FX question shares no
words at all with either document. On a real corpus it is narrower, which is why Lab 6.5 measures
the floor instead of arguing about it.

**The embedding is the part you did not have to build.** `LabEmbeddings` matches on shared words,
so it misses *&ldquo;can we push a large payment between our own entities&rdquo;* &mdash; a question that
means section 3.2 exactly and shares almost no words with it. A trained embedding gets that one
right. What it would *not* change is anything else you did here: top-k still returns k, a badly cut
chunk still cannot be reassembled, and an unfiltered index still returns things the reader should
not see. Those are the parts you build, and they are the rest of this module.

In [ ]:
score()

## Your turn

1. Give `split_by_size` a `chunk_overlap` of 40 and re-run the Section 1 checks. Does overlap
   actually reattach the exception to its rule, or does it just make the failure rarer and
   harder to find?
2. Ask `search` the semantic question &mdash; *&ldquo;can we push a large payment between our own
   entities&rdquo;* &mdash; and look at where 3.2 ranks. Now add the word &ldquo;intra-group&rdquo; to the query.
   That gap is the whole of Lab 6.2's third section.
3. Set `floor` to 0.1, 0.2 and 0.4 in turn and record, for both kinds of question, whether you got
   an answer and whether it was right. That table is the beginning of Lab 6.5.